# Three-dimensional Darcy: local problems, processes and scalability

This tutorial writes an anisotropic multiscale Darcy problem directly in UFL,
then declares its local and global MHM equations. We compare the physical
pressure and Darcy flux with an independently assembled conforming Q1 method.
The union of local fine meshes and the conforming mesh have the same fine
hexahedra; their global approximation spaces differ.

Follow the small demonstration first. A separate configuration enables larger
strong and weak studies. The published record can be inspected without rerunning
those workloads. Timings include complete workflows, and physical validation
runs after the timer stops.

Run this notebook from the checked Pixi environment:

```bash
pixi run --locked -e introduction python scripts/run_notebooks.py \
    notebooks/introduction/darcy_3d_parallel_scalability.ipynb --timeout 3600
```

The optional AmgX route requires its native accelerator environment. The CPU
example does not import accelerator packages.


In [ ]:
from __future__ import annotations

import os
from pathlib import Path

for variable in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "BLIS_NUM_THREADS"):
    os.environ[variable] = "1"

import ast
import atexit
import hashlib
import importlib.util
import json
import math
import platform
import random
import sys
import tempfile
import time
from dataclasses import dataclass, replace
from importlib.metadata import version
from itertools import product
from typing import Any

import matplotlib.pyplot as plt
from matplotlib import patheffects
import numpy as np
from threadpoolctl import threadpool_info, threadpool_limits

from pymhm.core.assembly import SolverConfig
from pymhm.core.equations import Equation, LocalEquations, compile_form
from pymhm.core.multiscale import MultiscaleProblem, assemble
from pymhm.execution.cpu import ExecutionConfig
from pymhm.fem.reference import tensor_lagrange_tabulation
from pymhm.meshes.hexahedron import HexMesh
from pymhm.io.provenance import current_source_manifest

ROOT = Path.cwd()
while not (ROOT / "pixi.toml").exists():
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Run this notebook within the PyMHM repository")
    ROOT = ROOT.parent
OUTPUT = ROOT / "build/introduction/darcy_3d_parallel_scalability"
OUTPUT.mkdir(parents=True, exist_ok=True)
PUBLISHED = ROOT / "benchmarks/results/execution/introduction-3d-20261004/results.json"
plt.rcParams.update({"figure.dpi": 110, "font.size": 10})


## 1. State the physical problem

On the box $\Omega_L=(0,L)\times(0,1)^2$, with integer $L$, solve

$$
\begin{aligned}
q&=-K(x)\nabla p, & \nabla\cdot q&=f,\\
p&=0 &&\text{on }\partial\Omega_L.
\end{aligned}
$$

Use the exact pressure and positive permeability

$$
\begin{aligned}
p_*(x)&=\prod_{i=1}^3\sin(\pi x_i),\\
K(x)&=m(x)D, &
m(x)&=\exp\!\left(\prod_{i=1}^3\sin(2\pi x_i/\varepsilon)\right),\\
D&=\begin{pmatrix}2&0.3&0.2\\0.3&1.5&0.1\\0.2&0.1&1\end{pmatrix}, &
\varepsilon&=0.1.
\end{aligned}
$$

The off-diagonal material entries require mixed pressure derivatives in the
source. Defining $f=\nabla\cdot(-K\nabla p_*)$ in UFL includes them automatically.
The NumPy formulas below provide an independent source and field evaluation.
Integer domain lengths preserve the exterior pressure data and physical
coefficient period when the box expands for weak scaling.


In [ ]:
DEFAULT_ANISOTROPY = np.array([[2.0, 0.3, 0.2], [0.3, 1.5, 0.1], [0.2, 0.1, 1.0]])


@dataclass(frozen=True)
class DarcyData:
    """Declare the medium period and constant symmetric positive anisotropy.

    The scalar medium multiplier is ``exp(prod(sin(2*pi*x_i/period)))``.
    Coordinates have shape ``(...,3)`` and tensor values ``(...,3,3)``.
    The exact pressure is ``prod(sin(pi*x_i))``. Integer domain lengths in x
    retain homogeneous exterior pressure and a fixed physical medium scale.
    """

    period: float = 0.1
    anisotropy: tuple[tuple[float, ...], ...] = tuple(map(tuple, DEFAULT_ANISOTROPY))

    def __post_init__(self) -> None:
        """Reject nonpositive periods and nonsymmetric or nonpositive material tensors."""
        matrix = np.asarray(self.anisotropy)
        if not np.isfinite(self.period) or self.period <= 0:
            raise ValueError("period must be finite and positive")
        if (matrix.shape != (3, 3) or not np.isfinite(matrix).all()
                or not np.array_equal(matrix, matrix.T) or np.linalg.eigvalsh(matrix)[0] <= 0):
            raise ValueError("anisotropy must be a finite symmetric positive definite 3 by 3 tensor")


def medium(points: np.ndarray, data: DarcyData) -> np.ndarray:
    """Evaluate the scalar multiplier, whose bounds are ``exp(-1)`` and ``exp(1)``."""
    return np.exp(np.prod(np.sin((2 * math.pi / data.period) * points), axis=-1))


def permeability(points: np.ndarray, data: DarcyData) -> np.ndarray:
    """Return the full physical diffusion tensor ``m(x)*D`` at every point."""
    return medium(points, data)[..., None, None] * np.asarray(data.anisotropy)


def exact_pressure(points: np.ndarray) -> np.ndarray:
    """Return the scalar exact pressure, with homogeneous data on integer boxes."""
    return np.prod(np.sin(math.pi * points), axis=-1)


def exact_gradient(points: np.ndarray) -> np.ndarray:
    """Differentiate pressure analytically in physical x, y and z coordinates."""
    sine, cosine = np.sin(math.pi * points), np.cos(math.pi * points)
    return math.pi * np.stack([cosine[..., i] * np.prod(sine[..., [j for j in range(3) if j != i]], axis=-1)
                               for i in range(3)], axis=-1)


def exact_hessian(points: np.ndarray) -> np.ndarray:
    """Return all physical pressure second derivatives, including mixed terms."""
    sine, cosine = np.sin(math.pi * points), np.cos(math.pi * points)
    result = np.empty((*points.shape[:-1], 3, 3))
    for i in range(3):
        for j in range(3):
            result[..., i, j] = (-math.pi**2 * np.prod(sine, axis=-1) if i == j else
                math.pi**2 * cosine[..., i] * cosine[..., j] * sine[..., 3 - i - j])
    return result


def exact_flux(points: np.ndarray, data: DarcyData) -> np.ndarray:
    """Return Darcy flux ``-K grad(p_exact)``, preserving all three components."""
    return -np.einsum("...ij,...j->...i", permeability(points, data), exact_gradient(points))


def source(points: np.ndarray, data: DarcyData) -> np.ndarray:
    """Apply minus divergence with independent analytic material and pressure derivatives."""
    omega = 2 * math.pi / data.period
    sine, cosine = np.sin(omega * points), np.cos(omega * points)
    multiplier = medium(points, data)
    grad_multiplier = omega * multiplier[..., None] * np.stack(
        [cosine[..., i] * np.prod(sine[..., [j for j in range(3) if j != i]], axis=-1)
         for i in range(3)], axis=-1)
    matrix = np.asarray(data.anisotropy)
    return (-multiplier * np.einsum("ij,...ij->...", matrix, exact_hessian(points))
            - np.einsum("...i,ij,...j->...", grad_multiplier, matrix, exact_gradient(points)))


The following function is the entire UFL definition of the physical data.
`SpatialCoordinate`, `grad`, `dot` and `div` follow the mathematical expressions
above. No Darcy-specific problem constructor is required.


In [ ]:
def ufl_data(domain: Any, data: DarcyData) -> tuple[Any, Any, Any, Any]:
    """Declare exact pressure, tensor, physical flux and source directly in UFL."""
    import ufl

    x = ufl.SpatialCoordinate(domain)
    pressure = ufl.sin(math.pi * x[0]) * ufl.sin(math.pi * x[1]) * ufl.sin(math.pi * x[2])
    omega = 2 * math.pi / data.period
    multiplier = ufl.exp(ufl.sin(omega * x[0]) * ufl.sin(omega * x[1]) * ufl.sin(omega * x[2]))
    tensor = multiplier * ufl.as_matrix(data.anisotropy)
    flux = -ufl.dot(tensor, ufl.grad(pressure))
    forcing = ufl.div(flux)
    return pressure, tensor, flux, forcing


The next check differentiates the independently evaluated exact flux by centered differences. It checks the source sign and full tensor before solving a problem.


In [ ]:
def verify_source(data: DarcyData = DarcyData()) -> dict[str, float]:
    """Check analytic source independently by centered differences of physical exact flux."""
    points = np.array([[0.237, 0.419, 0.561], [0.613, 0.728, 0.137], [0.832, 0.147, 0.349]])
    step, divergence = 2e-6, np.zeros(len(points))
    for axis in range(3):
        shift = np.eye(3)[axis] * step
        divergence += (exact_flux(points + shift, data)[:, axis] - exact_flux(points - shift, data)[:, axis]) / (2 * step)
    forcing = source(points, data)
    np.testing.assert_allclose(forcing, divergence, rtol=3e-8, atol=3e-8)
    return {"maximum_source_absolute_difference": float(np.max(np.abs(forcing - divergence))),
            "anisotropy_minimum_eigenvalue": float(np.linalg.eigvalsh(data.anisotropy)[0]),
            "anisotropy_maximum_eigenvalue": float(np.linalg.eigvalsh(data.anisotropy)[-1])}

source_control = verify_source()
source_control


## 2. Separate the macro mesh and local resolution

Four macrohexahedra per unit direction give $4^3=64$ local tasks on the unit
cube. Each macrocell contains $r^3$ conforming Q1 fine hexahedra, where
$r=n/4$. Thus $n=64,96,128$ gives 262,144, 884,736 and 2,097,152 fine elements.
The classical Q1 method uses exactly that structured fine grid.

Here `Q1` is a tensor-product space. An unsplit quadrilateral face has four
Q1 nodal modes; it has nine modes for Q2. The local volume degree remains one
when the trace degree changes. A richer trace can reduce skeleton error,
but it changes the global discretization and must be held fixed within a
strong-scaling study.

`HexMesh` owns shared face indices and normal signs. This Cartesian example
uses increasing physical tangent coordinates on every face, so neighboring
cells use the same face basis. Arbitrary mapped hexahedra additionally require
the mesh's face coordinate transformations.


In [ ]:
def macro_mesh(count: int = 4, length: int = 1) -> HexMesh:
    """Make Cartesian macrohexahedra with fixed physical width 1/count in all axes."""
    nx, ny, nz = count * length, count, count
    points = np.array(list(product(np.linspace(0, length, nx + 1),
                                  np.linspace(0, 1, ny + 1), np.linspace(0, 1, nz + 1))))
    corners = tuple(product((0, 1), repeat=3))
    cells = np.array([[(i+a)*(ny+1)*(nz+1)+(j+b)*(nz+1)+k+c
                       for a,b,c in corners] for i,j,k in product(range(nx),range(ny),range(nz))])
    return HexMesh(points, cells)


## 3. Write one local problem

Let $V_h(T)$ be the local Q1 space and let $\lambda$ represent Darcy normal flux
in a fixed orientation on each macroface. Its outward value on cell $T$ is
$s_{T,F}\lambda_F$. Integration by parts gives

$$
\begin{aligned}
a_T(p_T,v)+b_T(v,\lambda)&=(f,v)_T,\\
a_T(p,v)&=\int_T K\nabla p\cdot\nabla v,\\
b_T(v,\lambda)&=\sum_{F\subset\partial T}s_{T,F}\int_F v\lambda_F.
\end{aligned}
$$

The operator has the constant local kernel $Z_T=1$. Source and trace responses
are selected with the physical moment $\int_T p\,dx=0$; a separate retained
constant amplitude supplies the physical cell mean in the final field.

The provider below first writes `a` and `L` in UFL. Its remaining lines adapt
native volume and face integrals into `LocalEquations`. Basix supplies the
quadrilateral trace tabulation; PyMHM's generic owners perform constrained
solves, condensation, global contributions and reconstruction. The provider
returns owned coefficient arrays rather than live native backend objects.


In [ ]:
@dataclass(frozen=True)
class LocalProvider:
    """Build the declared Q1 Neumann forms and oriented Qk face columns in a worker."""

    macro: HexMesh
    refinement: int
    data: DarcyData = DarcyData()
    trace_degree: int = 1
    quadrature_degree: int = 8

    def __call__(self, cell: int) -> LocalEquations:
        """Return A p+B lambda=f and -B.T p=0, retaining the constant local mode."""
        import dolfinx
        import ufl
        from mpi4py import MPI

        started = time.perf_counter()
        corners = self.macro.points[self.macro.cells[cell]]
        lower, upper = corners.min(axis=0), corners.max(axis=0)
        domain = dolfinx.mesh.create_box(MPI.COMM_SELF, [lower, upper],
            [self.refinement]*3, cell_type=dolfinx.mesh.CellType.hexahedron)
        V = dolfinx.fem.functionspace(domain, ("Lagrange", 1))
        p, v = ufl.TrialFunction(V), ufl.TestFunction(V)
        _, tensor, _, forcing = ufl_data(domain, self.data)
        dx = ufl.Measure("dx", domain=domain, metadata={"quadrature_degree": self.quadrature_degree})
        a = ufl.inner(ufl.dot(tensor, ufl.grad(p)), ufl.grad(v))*dx
        L = forcing*v*dx
        A, f, moment = compile_form(a), compile_form(L), compile_form(v*dx)
        coordinates = V.tabulate_dof_coordinates()[:, :3].copy()
        facets, markers = [], []
        for side in range(6):
            axis, endpoint = divmod(side, 2)
            value = (lower, upper)[endpoint][axis]
            located = dolfinx.mesh.locate_entities_boundary(domain, 2,
                lambda x, axis=axis, value=value: np.isclose(x[axis],value,rtol=0,atol=1e-12))
            facets.extend(located.tolist())
            markers.extend([side+1]*len(located))
        ordering = np.argsort(facets)
        tags = dolfinx.mesh.meshtags(domain, 2, np.asarray(facets,dtype=np.int32)[ordering],
                                   np.asarray(markers,dtype=np.int32)[ordering])
        ds = ufl.Measure("ds", domain=domain, subdomain_data=tags,
                         metadata={"quadrature_degree": 2*self.trace_degree+2})
        T = V if self.trace_degree == 1 else dolfinx.fem.functionspace(domain,("Lagrange",self.trace_degree))
        trace_lift = dolfinx.fem.Function(T)
        trace_points = T.tabulate_dof_coordinates()[:, :3]
        reference_nodes = np.array(list(product(np.linspace(0,1,self.trace_degree+1),repeat=2)))
        width = len(reference_nodes)
        B = np.empty((len(coordinates), 6*width))
        for side in range(6):
            axis = side//2
            tangents = [d for d in range(3) if d != axis]
            uv = np.clip((trace_points[:,tangents]-lower[tangents])/(upper-lower)[tangents],0,1)
            values = tensor_lagrange_tabulation("quadrilateral",self.trace_degree,uv,
                                              nodes=reference_nodes,nderiv=0)[0]
            face_form = dolfinx.fem.form(trace_lift*v*ds(side+1))
            for index in range(width):
                trace_lift.x.array[:] = values[:,index]
                vector = dolfinx.fem.assemble_vector(face_form)
                B[:,side*width+index] = self.macro.signs[cell,side]*vector.array
        dofs = np.concatenate([face*width+np.arange(width) for face in self.macro.cell_faces[cell]])
        completed = time.perf_counter()
        return LocalEquations(a=A,L=f,b=B,c=-B.T,dofs=dofs,
            kernel=np.ones((len(coordinates),1)),moments=moment[:,None],
            metadata={"coordinates":coordinates,"bounds":np.vstack((lower,upper)),
                      "refinement":self.refinement,"fine_cells":self.refinement**3,
                      "local_assembly_seconds":completed-started,"physical_moment":moment})


A local audit checks the declared constant, the physical volume moment and
the signed face integrals. For Q1 or Q2 nodal traces on a rectangle, the
integrals of individual nodal modes are not generally equal; compare their
**sum**, which integrates the partition of unity. Local volume refinement must
also provide sufficient independent boundary DOFs for the selected trace.


In [ ]:
def audit_local(provider: LocalProvider, cell: int = 0) -> dict[str, float]:
    """Check constant-kernel, physical-volume and oriented face partition identities."""
    equations = provider(cell)
    matrix, coupling = equations.a, equations.b
    ones = np.ones(matrix.shape[0])
    lower, upper = equations.metadata["bounds"]
    lengths = upper - lower
    width = (provider.trace_degree + 1) ** 2
    kernel_action = float(np.linalg.norm(matrix @ ones))
    matrix_scale = float(np.max(np.asarray(abs(matrix).sum(axis=1))))
    assert kernel_action <= 1e-11 * matrix_scale * np.sqrt(len(ones))
    moment_volume = float(np.sum(equations.moments))
    np.testing.assert_allclose(moment_volume, np.prod(lengths), rtol=1e-12, atol=1e-14)
    integrals = ones @ coupling
    for side in range(6):
        area = float(np.prod(lengths[np.arange(3) != side // 2]))
        np.testing.assert_allclose(np.sum(integrals[side*width:(side+1)*width]),
                                   provider.macro.signs[cell, side] * area, rtol=1e-12, atol=1e-14)
    return {"constant_kernel_action_norm": kernel_action, "physical_volume": moment_volume,
            "trace_column_rank": int(np.linalg.matrix_rank(coupling))}


## 4. Make the displayed provider importable for processes

Python `spawn` gives each worker a fresh interpreter. The next cell copies the
literal definitions already executed above into a temporary importable module,
including the dataclass decorator and imports of generic package owners. The
module contains no hidden physical model. Its hash records the actual source
sent to the workers.

Each worker constructs and condenses its own macrocell. Only the coordinator
accumulates shared global entries, in macrocell order. `pipeline=True` uses a
bounded rolling window; `batch_size=workers` limits outstanding work. Native
libraries use one thread per worker.


In [ ]:
def literal_definitions(names: tuple[str, ...]) -> str:
    """Select exact executed IPython definitions and the material-tensor assignment."""
    selected: dict[str, str] = {}
    tensor_assignment = None
    for text in get_ipython().history_manager.input_hist_raw:
        try:
            tree = ast.parse(text)
        except SyntaxError:
            continue
        lines = text.splitlines()
        for node in tree.body:
            if isinstance(node, (ast.FunctionDef, ast.ClassDef)) and node.name in names:
                begin = min([node.lineno, *(item.lineno for item in node.decorator_list)])
                selected[node.name] = "\n".join(lines[begin-1:node.end_lineno])
            elif isinstance(node, ast.Assign) and any(
                isinstance(target, ast.Name) and target.id == "DEFAULT_ANISOTROPY"
                for target in node.targets
            ):
                tensor_assignment = ast.get_source_segment(text, node)
    if set(selected) != set(names) or tensor_assignment is None:
        raise ValueError("Execute the displayed data and provider definitions before export")
    return "\n\n\n".join([tensor_assignment, *(selected[name] for name in names)])


WORKER_IMPORTS = """from __future__ import annotations
from dataclasses import dataclass
from itertools import product
import math
import time
from typing import Any
import numpy as np
from pymhm.core.equations import LocalEquations, compile_form
from pymhm.fem.reference import tensor_lagrange_tabulation
from pymhm.meshes.hexahedron import HexMesh
"""
export_started = time.perf_counter()
worker_source = WORKER_IMPORTS + "\n" + literal_definitions(("DarcyData", "ufl_data", "LocalProvider")) + "\n"
worker_sha256 = hashlib.sha256(worker_source.encode()).hexdigest()
spawn_directory = tempfile.TemporaryDirectory(prefix="pymhm_darcy3d_spawn_")
atexit.register(spawn_directory.cleanup)
module_name = "pymhm_darcy3d_" + worker_sha256[:16]
module_path = Path(spawn_directory.name) / (module_name + ".py")
module_path.write_text(worker_source)
sys.path.insert(0, spawn_directory.name)
specification = importlib.util.spec_from_file_location(module_name, module_path)
worker_module = importlib.util.module_from_spec(specification)
sys.modules[module_name] = worker_module
specification.loader.exec_module(worker_module)
SpawnLocalProvider = worker_module.LocalProvider
export_import_seconds = time.perf_counter() - export_started
(OUTPUT / "literal_worker.py").write_text(worker_source)
print({"worker_source_sha256": worker_sha256, "one_time_export_import_seconds": export_import_seconds})


## 5. Declare the global problem and complete timer

Pressure continuity is tested weakly on interior faces; exterior pressure data
enter the same global pairing. Homogeneous exterior pressure gives a zero
right-hand side. In the matrix convention used here, this pairing is `c=-b.T`.
The generic global assembly includes the local constant compatibility rows.

$$
\begin{aligned}
A_T p_T+B_T\lambda_T&=f_T,\\
\sum_T(-B_T^{\mathsf T}p_T)&=0,\\
p_T&=R_Tf_T-R_TB_T\lambda_T+E_T\alpha_T.
\end{aligned}
$$

`Equation(0, 0)` supplies no additional global term. One retained coordinate is
allocated per macrocell. There is **no global mean-zero pressure gauge**: full
Dirichlet data determine the physical pressure, whose unit-cube exact mean is
$(2/\pi)^3$. The zero-mean condition applies only to the local responses.

AMG acts on the projected, pinned SPD local diffusion operator. The package
owner restores the physical moments and checks the original equations. A hierarchy
built for a solver call is reused across its right-hand sides; original-row
refinement remains inside the solver owner and timing.
The hybrid global system retains its generic sparse direct solver; an elliptic
AMG preset is not applied to this coupled system.

The timer starts before macro geometry and stops after complete field
reconstruction. Pool startup, child imports, native assembly, response
serialization, ordered accumulation, synchronization and pool shutdown remain
in the assembly interval. Every call builds fresh operators and solver data.


In [ ]:
def run_mhm(fine_n: int, *, workers: int = 1, backend: str = "process", solver: str = "scipy",
            length: int = 1, trace_degree: int = 1, macro_count: int = 4,
            quadrature_degree: int = 8) -> tuple[dict[str, Any], Any, Any, HexMesh]:
    """Time a fresh provider, executor, local/global assembly and complete reconstruction."""
    assert fine_n % macro_count == 0
    with threadpool_limits(1):
        started = time.perf_counter()
        macro = macro_mesh(macro_count,length)
        provider_type = LocalProvider if backend == "serial" else SpawnLocalProvider
        provider = provider_type(macro,fine_n//macro_count,trace_degree=trace_degree,
                                 quadrature_degree=quadrature_degree)
        width = (trace_degree+1)**2
        problem = MultiscaleProblem(Equation(0,0),provider,range(len(macro.cells)),
                                   len(macro.faces)*width,(1,)*len(macro.cells))
        prepared = time.perf_counter()
        system = assemble(problem,execution=ExecutionConfig(backend,workers,native_threads=1,
                          batch_size=workers,pipeline=True),
                          solvers=SolverConfig(local_solver=solver,global_solver="scipy"))
        assembled = time.perf_counter()
        solution = system.solve()
        finished = time.perf_counter()
        record = {"method":"MHM", "fine_n":fine_n,"length":length,"workers":workers,
            "backend":backend,"local_solver":solver,"trace_degree":trace_degree,
            "setup":prepared-started,"local_and_global_assembly":assembled-prepared,
            "global_solve_reconstruction":finished-assembled,"total":finished-started,
            "fine_cells":fine_n**3*length,"macro_cells":len(macro.cells),
            "local_nodal_dofs":(fine_n//macro_count+1)**3,"trace_dofs":problem.trace_size,
            "global_dofs":system.matrix.shape[0],"global_relative_residual":float(solution.residual),
            "local_assembly_seconds_sum":sum(r['local_assembly_seconds']
                    for r in system.local_metadata),"assembly_quadrature_degree":quadrature_degree}
    return record,system,solution,macro


## 6. Assemble the classical reference independently

The conforming method solves

$$
\int_{\Omega_L}K\nabla p_h\cdot\nabla v_h\,dx
=\int_{\Omega_L}f v_h\,dx,
\qquad p_h,v_h\in V_h\cap H^1_0(\Omega_L).
$$

The reference uses DOLFINx Q1 geometry and UFL volume forms on the matched
structured fine grid. Homogeneous exterior values are strongly eliminated.
The reduced matrix is SPD, with a fresh smoothed-aggregation AMG hierarchy and
CG solve. The constant AMG candidate is a near-nullspace vector; it does not
add a physical pressure gauge. The original reduced equations must satisfy
`rtol=1e-10` without changing the source or tolerance.


In [ ]:
@dataclass
class NativeState:
    """Keep native functions in their original executed coordinates for untimed norms."""

    domain: Any
    space: Any
    pressure: Any
    data: DarcyData
    grid: tuple[int, int, int]
    length: int
    gathered_coefficients: np.ndarray | None = None


def solve_classical(
    fine_n: int,
    *,
    data: DarcyData = DarcyData(),
    solver: str = "pyamg",
    length: int = 1,
    quadrature_degree: int = 8,
) -> tuple[dict[str, Any], NativeState]:
    """Time fresh native Q1 geometry, UFL assembly, boundary elimination and solve.

    Imports and JIT warmup belong to a separate explicitly recorded warmup.
    Native assembly, sparse-copy transfer and fresh AMG setup remain in every
    measured run. No factorization or AMG hierarchy is reused between runs.
    The physical relative residual is checked on the original reduced matrix.
    """
    import dolfinx
    import ufl
    from mpi4py import MPI
    from pymhm.core.equations import compile_form
    from pymhm.linalg.linear import solve_linear

    if fine_n < 2 or isinstance(fine_n, bool) or length < 1 or isinstance(length, bool):
        raise ValueError("fine_n and length must be positive integer counts, with fine_n at least two")
    grid = fine_n * length, fine_n, fine_n
    with threadpool_limits(1):
        started = time.perf_counter()
        domain = dolfinx.mesh.create_box(
            MPI.COMM_SELF, [np.zeros(3), np.array([float(length), 1.0, 1.0])],
            grid, cell_type=dolfinx.mesh.CellType.hexahedron)
        V = dolfinx.fem.functionspace(domain, ("Lagrange", 1))
        p, v = ufl.TrialFunction(V), ufl.TestFunction(V)
        _, tensor, _, forcing = ufl_data(domain, data)
        dx = ufl.Measure("dx", domain=domain, metadata={"quadrature_degree": quadrature_degree})
        a = ufl.inner(ufl.dot(tensor, ufl.grad(p)), ufl.grad(v)) * dx
        L = forcing * v * dx
        coordinates = V.tabulate_dof_coordinates()[:, :3]
        upper = np.array([length, 1, 1])
        boundary = np.any(np.isclose(coordinates, 0, rtol=0, atol=1e-13)
                          | np.isclose(coordinates, upper, rtol=0, atol=1e-13), axis=1)
        free = np.flatnonzero(~boundary)
        prepared = time.perf_counter()
        matrix, load = compile_form(a), compile_form(L)
        reduced, rhs = matrix[free][:, free], load[free]
        assembled = time.perf_counter()
        coefficients = np.zeros(len(coordinates))
        coefficients[free] = solve_linear(
            reduced, rhs, solver=solver, rtol=1e-10, atol=0,
            maxiter=500 if solver == "pyamg" else None,
            near_nullspace=np.ones((len(free), 1)) if solver == "pyamg" else None,
            refinement_precision="double", refinement_steps=2, equilibration="none")
        pressure = dolfinx.fem.Function(V)
        pressure.x.array[:] = coefficients
        completed = time.perf_counter()
        residual = float(np.linalg.norm(reduced @ coefficients[free] - rhs) / np.linalg.norm(rhs))
        assert residual <= 1e-10
        ncells = domain.topology.index_map(3).size_local
        assert ncells == math.prod(grid) and len(coefficients) == math.prod(n + 1 for n in grid)
        nnz = int(reduced.nnz)
    return ({"setup": prepared - started, "assembly_transfer_elimination": assembled - prepared,
             "solve_reconstruct": completed - assembled, "total": completed - started,
             "equation_relative_L2_residual": residual, "fine_grid": list(grid),
             "fine_cells": ncells, "global_algebraic_size": len(free), "matrix_nnz": nnz,
             "solver": solver, "assembly_quadrature_degree": quadrature_degree,
             "timing_scope": "fresh native setup, assembly/copy/BC elimination, solver setup/solve, complete field reconstruction"},
            NativeState(domain, V, pressure, data, grid, length))


For an equal-CPU-budget comparison, the independent conforming method can
instead use distributed PETSc CG/GAMG. Its rank barriers and maximum rank times
include partitioning, assembly, fresh hierarchy construction and reconstruction.
The following definition is shown explicitly; calling it from this notebook's
single-rank kernel uses one rank. A multi-rank acquisition must run with an
MPI communicator containing the reported number of ranks.


In [ ]:
def solve_classical_mpi(
    fine_n: int,
    *,
    comm: Any,
    data: DarcyData = DarcyData(),
    length: int = 1,
    quadrature_degree: int = 8,
) -> tuple[dict[str, Any], NativeState]:
    """Build and solve independent conforming Q1 with distributed PETSc CG/GAMG.

    Every rank uses one native thread. Collective barriers and rank maxima
    account for synchronization, mesh partitioning, assembly, a fresh AMG
    hierarchy and complete distributed field reconstruction. The constant
    near-nullspace is an AMG candidate, not a pressure gauge or exact nullspace.
    All manually created PETSc resources are destroyed before returning.
    """
    import dolfinx
    import dolfinx.fem.petsc
    import ufl
    from mpi4py import MPI
    from petsc4py import PETSc

    grid = fine_n * length, fine_n, fine_n
    matrix = load = ksp = candidate = defect = None
    with threadpool_limits(1):
        comm.Barrier()
        started = time.perf_counter()
        domain = dolfinx.mesh.create_box(
            comm, [np.zeros(3), np.array([float(length), 1.0, 1.0])], grid,
            cell_type=dolfinx.mesh.CellType.hexahedron)
        V = dolfinx.fem.functionspace(domain, ("Lagrange", 1))
        p, v = ufl.TrialFunction(V), ufl.TestFunction(V)
        _, tensor, _, forcing = ufl_data(domain, data)
        dx = ufl.Measure("dx", domain=domain, metadata={"quadrature_degree": quadrature_degree})
        a = dolfinx.fem.form(ufl.inner(ufl.dot(tensor, ufl.grad(p)), ufl.grad(v)) * dx)
        L = dolfinx.fem.form(forcing * v * dx)
        domain.topology.create_connectivity(2, 3)
        facets = dolfinx.mesh.exterior_facet_indices(domain.topology)
        dofs = dolfinx.fem.locate_dofs_topological(V, 2, facets)
        bc = dolfinx.fem.dirichletbc(PETSc.ScalarType(0), dofs, V)
        pressure = dolfinx.fem.Function(V)
        comm.Barrier()
        prepared = time.perf_counter()
        try:
            matrix = dolfinx.fem.petsc.assemble_matrix(a, bcs=[bc])
            matrix.assemble()
            load = dolfinx.fem.petsc.assemble_vector(L)
            dolfinx.fem.petsc.apply_lifting(load, [a], bcs=[[bc]])
            load.ghostUpdate(addv=PETSc.InsertMode.ADD_VALUES, mode=PETSc.ScatterMode.REVERSE)
            dolfinx.fem.petsc.set_bc(load, [bc])
            candidate = PETSc.NullSpace().create(constant=True, comm=comm)
            matrix.setNearNullSpace(candidate)
            comm.Barrier()
            assembled = time.perf_counter()
            ksp = PETSc.KSP().create(comm)
            ksp.setOperators(matrix)
            ksp.setType("cg")
            ksp.getPC().setType("gamg")
            ksp.setTolerances(rtol=1e-10, atol=0, max_it=500)
            ksp.setNormType(PETSc.KSP.NormType.UNPRECONDITIONED)
            ksp.setUp()
            comm.Barrier()
            hierarchy_prepared = time.perf_counter()
            ksp.solve(load, pressure.x.petsc_vec)
            pressure.x.scatter_forward()
            owned = V.dofmap.index_map.size_local * V.dofmap.index_map_bs
            coordinates = V.tabulate_dof_coordinates()[:owned, :3]
            lattice = np.rint(coordinates * fine_n).astype(np.int64)
            addresses = np.ravel_multi_index(lattice.T, tuple(n + 1 for n in grid))
            parts = comm.gather((addresses, pressure.x.array[:owned].copy()), root=0)
            canonical = None
            if comm.rank == 0:
                canonical = np.empty(math.prod(n + 1 for n in grid))
                indices = np.concatenate([part[0] for part in parts])
                assert len(indices) == len(canonical) and len(np.unique(indices)) == len(indices)
                canonical[indices] = np.concatenate([part[1] for part in parts])
            comm.Barrier()
            completed = time.perf_counter()
            defect = load.duplicate()
            matrix.mult(pressure.x.petsc_vec, defect)
            defect.axpy(-1, load)
            residual = float(defect.norm() / load.norm())
            reason, iterations = int(ksp.getConvergedReason()), int(ksp.getIterationNumber())
            assert reason > 0 and residual <= 1e-10, (reason, iterations, residual)
            ncells = int(comm.allreduce(domain.topology.index_map(3).size_local, op=MPI.SUM))
            ndofs = int(V.dofmap.index_map.size_global * V.dofmap.index_map_bs)
            assert ncells == math.prod(grid) and ndofs == math.prod(n + 1 for n in grid)
        finally:
            for resource in (defect, ksp, candidate, load, matrix):
                if resource is not None:
                    resource.destroy()
        elapsed = {"setup": prepared - started,
                   "assembly_transfer_elimination": assembled - prepared,
                   "hierarchy_setup": hierarchy_prepared - assembled,
                   "solve_gather_reconstruct": completed - hierarchy_prepared,
                   "solve_reconstruct": completed - assembled,
                   "total": completed - started}
        elapsed = {name: float(comm.allreduce(value, op=MPI.MAX)) for name, value in elapsed.items()}
    return ({**elapsed, "mpi_ranks": int(comm.size), "native_threads_per_rank": 1,
             "fine_grid": list(grid), "fine_cells": ncells, "global_algebraic_size": ndofs,
             "solver": "PETSc CG/GAMG", "iterations": iterations, "convergence_reason": reason,
             "equation_relative_L2_residual": residual,
             "assembly_quadrature_degree": quadrature_degree,
             "near_nullspace": "one constant candidate; no global pressure gauge",
             "timing_scope": "fresh distributed mesh/setup, synchronized assembly/BC elimination, fresh GAMG hierarchy/solve, ghost synchronization, canonical physical-node gather and complete reconstruction"},
            NativeState(domain, V, pressure, data, grid, length, canonical))


## 7. Validate physical fields outside performance timing

Integrate pressure and all components of the physical Darcy flux separately.
The MHM flux below is $-K\nabla p_h$ inside each local fine cell. It is a broken
gradient field; it is not an H(div) reconstruction and does not imply fine-cell
conservation. Macro conservation concerns the oriented trace moments.

The classical reference uses independent UFL integration. The local-field
integrator maps retained native nodes through integer lattice addresses and
uses Basix Q1 tabulation.
Weak-study errors are also divided by $\sqrt{|\Omega_L|}$ so domain growth does
not alone increase the reported norm. Adequate assembly and error quadrature,
reference refinement and serial/process agreement are separate controls.


In [ ]:
def classical_physical_errors(state: NativeState, *, quadrature_degree: int = 10) -> dict[str, float]:
    """Integrate physical p/q errors against exact fields with independent native UFL.

    Weak-scaling errors are normalized by sqrt(volume); dimensional absolute
    errors and exact field norms are retained separately. Physical flux is the
    broken conforming gradient field, without any H(div) reconstruction claim.
    This operation is scientific validation and is outside performance timing.
    """
    import dolfinx
    import ufl
    from mpi4py import MPI

    exact, tensor, flux, _ = ufl_data(state.domain, state.data)
    numerical_flux = -ufl.dot(tensor, ufl.grad(state.pressure))
    dx = ufl.Measure("dx", domain=state.domain, metadata={"quadrature_degree": quadrature_degree})
    dp, dq = state.pressure - exact, numerical_flux - flux
    integrands = {"pressure": dp**2, "flux": ufl.inner(dq, dq),
                  "exact_pressure": exact**2, "exact_flux": ufl.inner(flux, flux)}
    values = {name: math.sqrt(max(0.0, float(state.domain.comm.allreduce(
                  dolfinx.fem.assemble_scalar(dolfinx.fem.form(value * dx)), op=MPI.SUM))))
              for name, value in integrands.items()}
    pressure_integral = float(state.domain.comm.allreduce(
        dolfinx.fem.assemble_scalar(dolfinx.fem.form(state.pressure * dx)), op=MPI.SUM))
    exact_integral = (1 - math.cos(math.pi * state.length)) * 4 / math.pi**3
    volume = float(state.length)
    return {"pressure_L2": values["pressure"], "flux_L2": values["flux"],
            "pressure_L2_per_sqrt_volume": values["pressure"] / math.sqrt(volume),
            "flux_L2_per_sqrt_volume": values["flux"] / math.sqrt(volume),
            "pressure_relative_L2": values["pressure"] / values["exact_pressure"],
            "flux_relative_L2": values["flux"] / values["exact_flux"],
            "exact_pressure_L2": values["exact_pressure"], "exact_flux_L2": values["exact_flux"],
            "pressure_integral": pressure_integral, "exact_pressure_integral": exact_integral,
            "error_quadrature_degree": quadrature_degree}


def canonical_coefficients(state: NativeState) -> tuple[np.ndarray, np.ndarray]:
    """Export literal native Q1 coefficients into x/y/z lexicographic grid order.

    The transformation is only a coordinate permutation. No interpolation,
    fitted basis, numerical nullspace or smoothing enters this data contract.
    """
    if state.domain.comm.size != 1:
        raise ValueError("canonical grid export requires a serial native state")
    coordinates = state.space.tabulate_dof_coordinates()[:, :3]
    spacing = 1 / state.grid[1]
    indices = np.rint(coordinates / spacing).astype(np.int64)
    shape = tuple(n + 1 for n in state.grid)
    order = np.ravel_multi_index(indices.T, shape)
    assert len(np.unique(order)) == math.prod(shape)
    points, coefficients = np.empty_like(coordinates), np.empty(len(coordinates))
    points[order], coefficients[order] = coordinates, state.pressure.x.array
    return points, coefficients


In [ ]:
def mhm_physical_errors(system: Any, solution: Any, *, order: int = 5) -> dict[str,float]:
    """Integrate the broken physical pressure/flux through retained native nodal coordinates."""
    from pymhm.meshes.hexahedron import cube_quadrature
    reference, weights = cube_quadrature(order)
    corners = np.array(list(product((0,1),repeat=3)))
    phi, derivative = tensor_lagrange_tabulation("hexahedron",1,reference,nodes=corners,nderiv=1)
    pressure_square, flux_square, volume = 0.,0.,0.
    for metadata, field in zip(system.local_metadata,solution.fields,strict=True):
        nodes=metadata['coordinates']
        r=metadata['refinement']
        lower,upper=metadata['bounds']
        spacing=(upper-lower)/r
        lattice=np.rint((nodes-lower)/spacing).astype(np.int64)
        addresses=np.ravel_multi_index(lattice.T,(r+1,)*3)
        assert len(np.unique(addresses)) == (r+1)**3
        permutation=np.argsort(addresses)
        values=np.asarray(field)[permutation]
        cells=np.array([[(i+a)*(r+1)**2+(j+b)*(r+1)+k+c for a,b,c in corners]
                        for i,j,k in product(range(r),repeat=3)])
        for begin in range(0,len(cells),256):
            ids=cells[begin:begin+256]
            coefficients=values[ids]
            origins=nodes[permutation][ids[:,0]]
            points=origins[:,None,:]+reference[None,:,:]*spacing
            dp=coefficients@phi.T-exact_pressure(points)
            gradient=np.einsum('ti,qid->tqd',coefficients,derivative/spacing)
            dq=np.einsum('tqij,tqj->tqi',permeability(points,DarcyData()),
                         gradient-exact_gradient(points))
            determinant=float(np.prod(spacing))
            pressure_square+=determinant*float(np.einsum('q,tq,tq->',weights,dp,dp))
            flux_square+=determinant*float(np.einsum('q,tqd,tqd->',weights,dq,dq))
        volume+=float(np.prod(upper-lower))
    return {'pressure_L2':float(np.sqrt(pressure_square)), 'flux_L2':float(np.sqrt(flux_square)),
            'pressure_L2_per_sqrt_volume':float(np.sqrt(pressure_square/volume)),
            'flux_L2_per_sqrt_volume':float(np.sqrt(flux_square/volume)),'error_gauss_order':order}


## 8. Select a small demonstration or a larger campaign

The small default checks the executable formulation and illustrates the cost
categories. It has one timing sample per configuration, so its times are not
robust performance estimates. Its coarse fine mesh also underresolves the
material compared with the large campaign. Use the published resolved fields
and quadrature controls for quantitative accuracy conclusions.

Strong scaling fixes the full discretization and uses
$S(P)=T_{\mathrm{process},1}/T_{\mathrm{process},P}$ and $E(P)=S(P)/P$.
True serial MHM is a separate baseline because spawning one process has startup
and transfer costs. Weak scaling expands the x-domain to integer length $L=P$
while preserving local resolution, macro width, coefficient period and physical
data. There are $64P$ macrocell tasks, so local work per process is constant;
the global skeleton still grows.

The larger settings expose 96 and 128 fine elements per unit direction. Select
repetitions and worker counts for the actual machine. Compare CPU1 classical
AMG with the stated MHM process budget; that comparison alone is not an
equal-32-core comparison. Distributed classical records give their own rank
counts. GPU component records have their own timing scope.


In [ ]:
RUN_SMALL_REPRODUCTION = True
RUN_LARGE_CAMPAIGN = False
REPETITIONS = 1 if not RUN_LARGE_CAMPAIGN else 3
STRONG_FINE_COUNTS = (16,) if not RUN_LARGE_CAMPAIGN else (64, 96, 128)
WORKER_COUNTS = (1, 2) if not RUN_LARGE_CAMPAIGN else (1, 8, 16, 32)
WEAK_FINE_N = 16 if not RUN_LARGE_CAMPAIGN else 64
WEAK_WORKERS = (1, 2) if not RUN_LARGE_CAMPAIGN else (1, 2, 4, 8)
TRACE_DEGREE = 1
ASSEMBLY_QUADRATURE_DEGREE = 8
RANDOM_SEED = 731

reproduction = {"metadata": {"repetitions": REPETITIONS, "native_threads": 1,
    "trace_degree": TRACE_DEGREE, "assembly_quadrature_degree": ASSEMBLY_QUADRATURE_DEGREE,
    "worker_source_sha256": worker_sha256, "python": platform.python_version(),
    "source_manifest": current_source_manifest({
        "source_notebook": hashlib.sha256((ROOT / "notebooks/introduction/darcy_3d_parallel_scalability.ipynb").read_bytes()).hexdigest(),
        "pixi.lock": hashlib.sha256((ROOT / "pixi.lock").read_bytes()).hexdigest()}),
    "load_average": list(os.getloadavg()) if hasattr(os, "getloadavg") else None,
    "affinity": sorted(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else None,
    "versions": {name: version(name) for name in ("numpy", "scipy", "fenics-basix", "fenics-dolfinx", "pyamg")},
    "native_libraries": threadpool_info()}, "warmup": [], "strong": [], "weak": [], "classical": []}
last_mhm = None
last_classical = None


Warm each form and solver family before measured samples. This populates the
existing compiler cache; it does not clear that cache or measure a cold FFCx
compilation. A fresh process pool still starts for every measured process call,
so fresh interpreter imports and first-use initialization remain timed. Local
factorizations and AMG hierarchies are rebuilt for every sample.

Do not run unrelated project benchmarks concurrently. CPU affinity, NUMA first
touch, native thread budgets and background load can affect results. Thirty-two
physical cores differ from sixty-four SMT threads on a two-socket host.


In [ ]:
if RUN_SMALL_REPRODUCTION or RUN_LARGE_CAMPAIGN:
    controls = audit_local(LocalProvider(macro_mesh(), max(4, STRONG_FINE_COUNTS[0] // 4),
                            trace_degree=TRACE_DEGREE,
                            quadrature_degree=ASSEMBLY_QUADRATURE_DEGREE))
    reproduction["accuracy_controls"] = {"source": source_control, "local": controls}
    warmup_n = 16
    classical_warm, _ = solve_classical(warmup_n, quadrature_degree=ASSEMBLY_QUADRATURE_DEGREE)
    reproduction["warmup"].append({"method": "classical", **classical_warm})
    for workers in WORKER_COUNTS:
        record, _, _, _ = run_mhm(warmup_n, workers=workers, backend="process", solver="pyamg",
            trace_degree=TRACE_DEGREE, quadrature_degree=ASSEMBLY_QUADRATURE_DEGREE)
        reproduction["warmup"].append(record)

    rng = random.Random(RANDOM_SEED)
    for fine_n in STRONG_FINE_COUNTS:
        configurations = [("classical", 1), ("serial", 1),
                          *(("process", workers) for workers in WORKER_COUNTS)]
        for repetition in range(REPETITIONS):
            order = configurations.copy()
            rng.shuffle(order)
            for backend, workers in order:
                if backend == "classical":
                    record, state = solve_classical(fine_n, quadrature_degree=ASSEMBLY_QUADRATURE_DEGREE)
                    record.update({"method": "classical", "fine_n": fine_n, "workers": 1,
                                   "repetition": repetition, "physical_errors": classical_physical_errors(state)})
                    reproduction["classical"].append(record)
                    last_classical = state
                else:
                    record, system, solution, macro = run_mhm(fine_n, workers=workers, backend=backend,
                        solver="pyamg", trace_degree=TRACE_DEGREE,
                        quadrature_degree=ASSEMBLY_QUADRATURE_DEGREE)
                    record.update({"repetition": repetition,
                                   "physical_errors": mhm_physical_errors(system, solution, order=7)})
                    reproduction["strong"].append(record)
                    last_mhm = system, solution, macro
                print({"fine_n": fine_n, "backend": backend, "workers": workers,
                       "total_seconds": record["total"], "physical_errors": record["physical_errors"]})


The weak study below changes domain length, rather than refining the local mesh or changing the coefficient wavelength. Each workflow includes its larger global assembly and solve.


In [ ]:
if RUN_LARGE_CAMPAIGN:
    for repetition in range(REPETITIONS):
        order = list(WEAK_WORKERS)
        rng.shuffle(order)
        for workers in order:
            record, system, solution, _ = run_mhm(WEAK_FINE_N, workers=workers, backend="process",
                solver="pyamg", length=workers, trace_degree=TRACE_DEGREE,
                quadrature_degree=ASSEMBLY_QUADRATURE_DEGREE)
            record.update({"repetition": repetition,
                           "physical_errors": mhm_physical_errors(system, solution, order=7)})
            reproduction["weak"].append(record)

if RUN_SMALL_REPRODUCTION or RUN_LARGE_CAMPAIGN:
    (OUTPUT / "reproduction.json").write_text(json.dumps(reproduction, indent=2) + "\n")


## 9. Plot pressure and Darcy flux with the actual macro mesh

The sampling below evaluates each intersected local fine cell independently.
It retains both sides of macro interfaces and performs no intercell averaging.
Every exact, numerical and error panel overlays the same actual macro mesh.
Each panel receives a separate color scale.


In [ ]:
def slice_fields(system: Any, solution: Any, plane: float = 0.37) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Sample Q1 pressure and physical flux at independent cut fine-cell centroids."""
    corners = np.array(list(product((0, 1), repeat=3)))
    point_blocks, pressure_blocks, flux_blocks = [], [], []
    for metadata, field in zip(system.local_metadata, solution.fields, strict=True):
        lower, upper = metadata["bounds"]
        if not lower[2] < plane < upper[2]:
            continue
        nodes = metadata["coordinates"]
        r = metadata["refinement"]
        spacing = (upper-lower) / r
        lattice = np.rint((nodes-lower)/spacing).astype(np.int64)
        addresses = np.ravel_multi_index(lattice.T, (r+1,)*3)
        assert len(np.unique(addresses)) == (r+1)**3
        permutation = np.argsort(addresses)
        coordinates, values = nodes[permutation], np.asarray(field)[permutation]
        k = int((plane-lower[2]) / spacing[2])
        ids = np.array([[(i+a)*(r+1)**2+(j+b)*(r+1)+k+c for a,b,c in corners]
                        for i,j in product(range(r), repeat=2)])
        origins = coordinates[ids[:, 0]]
        reference = np.array([[0.5, 0.5, (plane-origins[0,2])/spacing[2]]])
        phi, derivative = tensor_lagrange_tabulation("hexahedron", 1, reference, nodes=corners, nderiv=1)
        points = origins + reference[0]*spacing
        pressure = (values[ids] @ phi.T)[:, 0]
        gradient = np.einsum("ti,id->td", values[ids], derivative[0]/spacing)
        flux = -np.einsum("tij,tj->ti", permeability(points, DarcyData()), gradient)
        point_blocks.append(points)
        pressure_blocks.append(pressure)
        flux_blocks.append(flux)
    return np.concatenate(point_blocks), np.concatenate(pressure_blocks), np.concatenate(flux_blocks)


def classical_at_points(state: NativeState, points: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Evaluate literal conforming Q1 coefficients in their coordinate-addressed cells."""
    _, coefficients = canonical_coefficients(state)
    n = state.grid[1]
    cells = np.floor(points*n).astype(np.int64)
    cells = np.minimum(cells, np.array(state.grid)-1)
    reference = points*n-cells
    corners = np.array(list(product((0,1), repeat=3)))
    ids = np.stack([np.ravel_multi_index((cells+corner).T, tuple(i+1 for i in state.grid))
                    for corner in corners], axis=1)
    phi, derivative = tensor_lagrange_tabulation("hexahedron", 1, reference, nodes=corners, nderiv=1)
    pressure = np.einsum("ti,ti->t", coefficients[ids], phi)
    gradient = np.einsum("ti,tid->td", coefficients[ids], derivative*n)
    flux = -np.einsum("tij,tj->ti", permeability(points, state.data), gradient)
    return pressure, flux


def plot_mhm_slice(system: Any, solution: Any, macro: HexMesh, classical: NativeState) -> list[Path]:
    """Compare exact/classical/local fields with distinct macro overlays and colorbars."""
    points, pressure, flux = slice_fields(system, solution)
    exact_p, exact_q = exact_pressure(points), exact_flux(points, DarcyData())
    classical_p, classical_q = classical_at_points(classical, points)
    families = {
        "pressure": ([("Exact pressure", exact_p), ("Classical Q1 pressure", classical_p),
                      ("MHM pressure", pressure), ("Material multiplier", medium(points, DarcyData())),
                      ("Classical pressure error", classical_p-exact_p), ("MHM pressure error", pressure-exact_p)]),
        "flux": ([("Exact Darcy flux magnitude", np.linalg.norm(exact_q, axis=1)),
                  ("Classical Darcy flux magnitude", np.linalg.norm(classical_q, axis=1)),
                  ("MHM Darcy flux magnitude", np.linalg.norm(flux, axis=1)),
                  ("Material multiplier", medium(points, DarcyData())),
                  ("Classical vector-flux error magnitude", np.linalg.norm(classical_q-exact_q, axis=1)),
                  ("MHM vector-flux error magnitude", np.linalg.norm(flux-exact_q, axis=1))])}
    destinations = []
    x_lines, y_lines = np.unique(macro.points[:,0]), np.unique(macro.points[:,1])
    for name, quantities in families.items():
        fig, axes = plt.subplots(2, 3, figsize=(13, 8), layout="constrained")
        shared_min = min(float(np.min(values)) for _, values in quantities[:3])
        shared_max = max(float(np.max(values)) for _, values in quantities[:3])
        for index, (ax, (title, values)) in enumerate(zip(axes.flat, quantities, strict=True)):
            options = {"vmin": shared_min, "vmax": shared_max} if index < 3 else {}
            artist = ax.scatter(points[:,0], points[:,1], c=values, s=35, marker="s", cmap="viridis", **options)
            fig.colorbar(artist, ax=ax, shrink=0.8)
            for x in x_lines:
                ax.axvline(x, color="black", linewidth=0.7,
                           path_effects=[patheffects.withStroke(linewidth=1.7, foreground="white")])
            for y in y_lines:
                ax.axhline(y, color="black", linewidth=0.7,
                           path_effects=[patheffects.withStroke(linewidth=1.7, foreground="white")])
            ax.set(title=title, xlabel="x", ylabel="y", aspect="equal",
                   xlim=(x_lines[0],x_lines[-1]), ylim=(y_lines[0],y_lines[-1]))
        fig.suptitle("Darcy fields at z = 0.37; black lines mark macrofaces")
        destination = OUTPUT / f"small_{name}_fields.png"
        fig.savefig(destination, dpi=160)
        plt.show()
        destinations.append(destination)
    return destinations


if last_mhm is not None and last_classical is not None:
    plot_mhm_slice(*last_mhm, last_classical)


## 10. Preserve the executed numerical basis with coefficients

A retained amplitude is meaningful only with its executed basis. The archive
below stores the source and trace lifts, actual retained matrix, trace map,
local coordinates and final coefficients for every macrocell. Reconstruction
uses the generic response owner before comparing to the saved field.
Archive writing and digest calculation occur outside the performance timer.


In [ ]:
def archive_fields(system: Any, solution: Any, destination: Path) -> dict[str, Any]:
    """Persist executed basis/lifts/geometry and verify reconstruction in their coordinates."""
    arrays: dict[str, np.ndarray] = {"global_trace": solution.trace}
    largest = 0.0
    basis_digests: dict[str, str] = {}
    for cell, (response, coefficients, field, metadata) in enumerate(zip(
            system.responses, solution.coarse, solution.fields, system.local_metadata, strict=True)):
        local_trace = solution.trace[response.problem.trace_dofs]
        for native_threads in (1, 2):
            with threadpool_limits(native_threads):
                replay = response.reconstruct(local_trace, coefficients)
                rotated = replace(response, coarse_vectors=-response.retained_basis)
                rotated_replay = rotated.reconstruct(local_trace, -coefficients)
                np.testing.assert_allclose(replay, field, rtol=1e-13, atol=1e-13)
                np.testing.assert_allclose(rotated_replay, field, rtol=1e-13, atol=1e-13)
                largest = max(largest, float(np.max(np.abs(replay-field), initial=0)),
                              float(np.max(np.abs(rotated_replay-field), initial=0)))
        basis = np.ascontiguousarray(response.retained_basis)
        header = json.dumps({"shape": basis.shape, "dtype": basis.dtype.str}, sort_keys=True).encode()
        basis_digests[str(cell)] = hashlib.sha256(header + basis.tobytes()).hexdigest()
        prefix = f"cell_{cell}_"
        for name, values in {"source": response.source, "trace_lifts": response.lifts,
                "retained_basis": response.retained_basis, "trace_dofs": response.problem.trace_dofs,
                "coarse_coefficients": coefficients, "field_coefficients": field,
                "coordinates": metadata["coordinates"], "bounds": metadata["bounds"],
                "physical_moments": response.problem.constraints, "declared_kernel": response.problem.kernel}.items():
            arrays[prefix+name] = np.asarray(values)
    np.savez_compressed(destination, **arrays)
    return {"path": destination.name, "sha256": hashlib.sha256(destination.read_bytes()).hexdigest(),
            "maximum_coefficient_replay_difference": largest, "cells": len(system.responses),
            "executed_basis_sha256": basis_digests, "replay_native_threads": [1, 2],
            "equivalent_retained_rotation": "sign reversal with matching coefficient reversal"}


if last_mhm is not None:
    reproduction["archive"] = archive_fields(last_mhm[0], last_mhm[1], OUTPUT / "fields_and_executed_bases.npz")
    (OUTPUT / "reproduction.json").write_text(json.dumps(reproduction, indent=2) + "\n")


## 11. Read the resolved published campaign

The public record reports actual measurements independently from this small
reproduction. Inspect its metadata, all timing samples, approximation spaces,
resource counts, field errors and numerical controls before comparing speedups.
Missing routes or failed numerical controls do not become estimated timings.
A GPU condensation component is not a complete CPU/GPU workflow measurement.

The two-socket workstation has 32 physical cores and 64 SMT threads, plus two
RTX A5000 GPUs. Resource counts, affinity, compiler initialization and native
library settings belong to each recorded acquisition. The CPU local solves are
independent, but startup, transfer, global assembly/solution and reconstruction
remain in the complete timing and can limit scaling.


In [ ]:
published = json.loads(PUBLISHED.read_text()) if PUBLISHED.exists() else None
if published is not None:
    print(json.dumps(published.get("metadata", {}), indent=2))
    print({section: len(published.get(section, []))
           for section in ("strong", "weak", "classical", "gpu")})
else:
    print("No published campaign record is installed; the small reproduction remains available.")


For repeated configurations, report medians and the full measured range. A
single sample remains a single sample. Group only records with matching
physical domain, fine grid, trace space, solver and execution resource count.
The utility below computes process self-speedups and weak efficiencies without
substituting a serial-method or classical time for the one-process denominator.


In [ ]:
def summarize_scaling(record: dict[str, Any]) -> list[dict[str, Any]]:
    """Summarize complete process samples at fixed strong discretization or weak local work."""
    summaries = []
    for section in ("strong", "weak"):
        groups: dict[tuple[Any, ...], list[float]] = {}
        for sample in record.get(section, []):
            if sample.get("backend") != "process":
                continue
            key = (sample.get("fine_n"), sample.get("trace_degree"),
                   sample.get("local_solver"), sample.get("workers"))
            groups.setdefault(key, []).append(float(sample["total"]))
        medians = {key: float(np.median(values)) for key, values in groups.items()}
        for key, values in sorted(groups.items()):
            denominator = medians.get((*key[:3], 1))
            workers = key[3]
            median = medians[key]
            ratio = None if denominator is None else denominator/median
            summaries.append({"study": section, "fine_n": key[0], "trace_degree": key[1],
                "local_solver": key[2], "workers": workers, "samples": len(values),
                "median_seconds": median, "minimum_seconds": min(values), "maximum_seconds": max(values),
                "speedup_or_weak_efficiency": ratio,
                "strong_efficiency": ratio/workers if ratio is not None and section == "strong" else None})
    return summaries


scaling_summary = summarize_scaling(published if published is not None else reproduction)
scaling_summary


In [ ]:
def plot_scaling(rows: list[dict[str, Any]]) -> Path | None:
    """Plot measured strong speedups and weak efficiencies with separate discretizations."""
    if not rows:
        return None
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), layout="constrained")
    families = sorted({(row["fine_n"],row["trace_degree"],row["local_solver"]) for row in rows})
    for family in families:
        for study, ax in zip(("strong", "weak"), axes, strict=True):
            selected = sorted((row for row in rows if row["study"] == study and
                (row["fine_n"],row["trace_degree"],row["local_solver"]) == family), key=lambda row: row["workers"])
            selected = [row for row in selected if row["speedup_or_weak_efficiency"] is not None]
            if selected:
                ax.plot([row["workers"] for row in selected],
                        [row["speedup_or_weak_efficiency"] for row in selected], "o-",
                        label=f"n={family[0]}, trace Q{family[1]}, {family[2]}")
    axes[0].set(title="Strong process self-speedup", xlabel="Processes", ylabel="T(1) / T(P)")
    axes[1].set(title="Weak efficiency", xlabel="Processes / box length", ylabel="T(1) / T(P)")
    for ax in axes:
        ax.grid(alpha=0.25)
        handles, labels = ax.get_legend_handles_labels()
        if handles:
            ax.legend(handles, labels, fontsize=8)
    destination = OUTPUT / "scaling_summary.png"
    fig.savefig(destination, dpi=160)
    plt.show()
    return destination


plot_scaling(scaling_summary)


### Local-resolution performance sweep and physical errors

With the macro mesh and Q1 trace fixed, changing the local fine resolution is
not a full MHM convergence study. The plot below compares its physical errors
with the independently refined conforming Q1 reference. A pressure-error floor
can remain even when the local volume discretization is finer. Classical
refinement rates are calculated from its own recorded errors, without assigning
those rates to the MHM sweep.


In [ ]:
def accuracy_rows(record: dict[str, Any]) -> list[dict[str, Any]]:
    """Select actual physical norms with separate classical/MHM spaces and solver labels."""
    rows: dict[tuple[Any, ...], dict[str, Any]] = {}
    for section in ("strong", "classical"):
        for sample in record.get(section, []):
            errors = sample.get("physical_errors")
            if not errors:
                continue
            fine_n = sample.get("fine_n", sample.get("fine_grid", [None, None])[1])
            if fine_n is None:
                continue
            label = (f"MHM / {sample.get('local_solver', 'unspecified')}" if section == "strong" else
                     f"Classical / {sample.get('solver', 'unspecified')} / {sample.get('mpi_ranks', 1)} ranks")
            key = label, fine_n, sample.get("trace_degree")
            rows.setdefault(key, {"label": label, "fine_n": fine_n, "pressure_L2": errors["pressure_L2"],
                                  "flux_L2": errors["flux_L2"]})
    return list(rows.values())


def plot_accuracy(rows: list[dict[str, Any]]) -> list[dict[str, Any]]:
    """Plot separately integrated errors and return observed classical refinement rates."""
    if not rows:
        return []
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), layout="constrained")
    classical_rates = []
    for label in sorted({row["label"] for row in rows}):
        selected = sorted((row for row in rows if row["label"] == label), key=lambda row: row["fine_n"])
        fine_counts = [row["fine_n"] for row in selected]
        for ax, field, title in zip(axes, ("pressure_L2", "flux_L2"),
                                    ("Pressure error", "Physical vector-flux error"), strict=True):
            ax.loglog(fine_counts, [row[field] for row in selected], "o-", label=label)
            ax.set(title=title, xlabel="Fine cells per unit direction", ylabel="L2 error")
            ax.grid(alpha=0.25)
        if label.startswith("Classical"):
            for first, second in zip(selected[:-1], selected[1:], strict=True):
                denominator = math.log(second["fine_n"]/first["fine_n"])
                classical_rates.append({"method": label, "from_n": first["fine_n"], "to_n": second["fine_n"],
                    "pressure_rate": math.log(first["pressure_L2"]/second["pressure_L2"])/denominator,
                    "flux_rate": math.log(first["flux_L2"]/second["flux_L2"])/denominator})
    for ax in axes:
        ax.legend(fontsize=8)
    fig.suptitle("Local-resolution performance sweep; macro mesh and MHM trace are fixed")
    fig.savefig(OUTPUT / "physical_accuracy_by_resolution.png", dpi=160)
    plt.show()
    return classical_rates


field_error_rows = accuracy_rows(published if published is not None else reproduction)
observed_classical_rates = plot_accuracy(field_error_rows)
observed_classical_rates


## 12. Optional local AMG on one or two GPUs

The same local equations can select
`SolverConfig(local_solver="amgx", global_solver="scipy")` instead of
`SolverConfig(local_solver="pyamg", global_solver="scipy")`. Both use the
original `LocalProblem` and shared Neumann projection/physical-moment owner.
The hybrid global problem continues to use sparse direct algebra.

For an explicit multiGPU component study, first save the original assembled
operators, loads, trace columns, declared kernel and physical moments. Loading
an archive below reconstructs that unchanged contract. Each GPU worker gets
one physical device, set before its first CuPy or AmgX import. It remains alive
for its shard of local operators and returns complete host `LocalResponse`
objects; each operator gets its own fresh hierarchy.

This measurement starts from already assembled operators. It includes fresh
spawn, imports, archive reads, native initialization, matrix/RHS transfers,
all solves, synchronization, cleanup, return serialization and pool shutdown.
It excludes FEM assembly and the global solve, so its speedup is a component
speedup. Do not add independently measured medians to invent a total time.
The optional section is disabled by default and requires the native AmgX and
CuPy environment with the selected GPUs available.


In [ ]:
from concurrent.futures import ProcessPoolExecutor
from contextlib import ExitStack
import multiprocessing as mp

from scipy import sparse
from pymhm.core.contracts import LocalProblem


def archive_local(system: Any, destination: Path, cell: int = 0) -> None:
    """Archive original Neumann coefficients and the exact executed kernel/moments."""
    problem = system.responses[cell].problem
    A = problem.matrix.tocsr()
    np.savez_compressed(destination,A_data=A.data,A_indices=A.indices,A_indptr=A.indptr,
        A_shape=A.shape,B=problem.coupling,f=problem.load,Z=problem.kernel,C=problem.constraints,
        trace_dofs=problem.trace_dofs)


def load_problem(path: Path) -> LocalProblem:
    """Load the executed CSR, face columns and physical moments without changing them."""
    with np.load(path, allow_pickle=False) as archive:
        operator = sparse.csr_matrix((archive['A_data'], archive['A_indices'], archive['A_indptr']), shape=tuple(archive['A_shape']))
        return LocalProblem(operator, archive['B'], archive['f'], archive['trace_dofs'], kernel=archive['Z'], constraints=archive['C'])


def original_rows(problem: LocalProblem, columns: np.ndarray) -> dict[str, Any]:
    """Independently fit the auxiliary moments and check every original KKT row."""
    rhs = np.column_stack((problem.load, problem.coupling))
    residual = rhs.astype(np.longdouble) - problem.matrix @ columns.astype(np.longdouble)
    constraints = problem.constraints.astype(np.longdouble)
    gram = np.asarray(constraints.T @ constraints, dtype=float)
    moments = np.linalg.solve(gram, np.asarray(constraints.T @ residual, dtype=float))
    residual -= constraints @ moments.astype(np.longdouble)
    gauge = constraints.T @ columns.astype(np.longdouble)
    row_norms = np.linalg.norm(np.asarray(residual, dtype=float), axis=0)
    gauge_norms = np.linalg.norm(np.asarray(gauge, dtype=float), axis=0)
    augmented_norms = np.hypot(row_norms, gauge_norms)
    rhs_norms = np.linalg.norm(rhs, axis=0)
    bounds = 1e-10 * rhs_norms
    assert np.all(augmented_norms <= bounds), (augmented_norms, bounds)
    return {
        'physical_absolute_residuals': row_norms.tolist(),
        'physical_relative_residuals': np.divide(row_norms, rhs_norms, out=np.zeros_like(row_norms), where=rhs_norms>0).tolist(),
        'augmented_relative_residuals': np.divide(augmented_norms, rhs_norms, out=np.zeros_like(augmented_norms), where=rhs_norms>0).tolist(),
        'physical_accepted_bounds': bounds.tolist(),
        'gauge_absolute_defects': np.max(np.abs(gauge), axis=0).astype(float).tolist(),
    }


The worker sets its device visibility before importing CUDA packages. The coordinator creates one single-worker executor per selected GPU, waits for every response and then joins every executor. Device-wide synchronization brackets the native work.


In [ ]:
def condense_shard(solver: str, slot: int, archives: tuple[str, ...]) -> tuple[list[tuple[str, Any]], dict[str, Any]]:
    """Own one fixed device or CPU shard in a fresh spawn process and return host data."""
    cp = None
    metadata: dict[str, Any] = {'solver': solver, 'slot': slot, 'operator_count': len(archives)}
    if solver == 'amgx':
        os.environ['CUDA_VISIBLE_DEVICES'] = str(slot)
        import cupy as cp
        assert cp.cuda.runtime.getDeviceCount() == 1
        cp.cuda.runtime.deviceSynchronize()
        properties = cp.cuda.runtime.getDeviceProperties(0)
        before_free, total = cp.cuda.runtime.memGetInfo()
        metadata.update({'physical_device': slot, 'visible_device': 0, 'visible_device_count': cp.cuda.runtime.getDeviceCount(), 'cuda_visible_devices': os.environ['CUDA_VISIBLE_DEVICES'], 'name': properties['name'].decode(), 'cuda_runtime': cp.cuda.runtime.runtimeGetVersion(), 'cuda_driver': cp.cuda.runtime.driverGetVersion(), 'cupy_version': cp.__version__, 'memory_total': total, 'free_memory_before_native_condensation': before_free})

    result = []
    try:
        for archive in archives:
            problem = load_problem(Path(archive))
            result.append((Path(archive).name, problem.condense(solver)))
    finally:
        if cp is not None:
            cp.cuda.runtime.deviceSynchronize()
    if cp is not None:
        after_free, _ = cp.cuda.runtime.memGetInfo()
        metadata['free_memory_after_native_cleanup'] = after_free
    return result, metadata


def run_gpu_component(archives: tuple[Path, ...], solver: str, slots: tuple[int, ...]) -> tuple[float, list[tuple[str, Any]], list[dict[str, Any]]]:
    """Include fresh spawn, import, load, native lifecycle, return and shutdown."""
    beginning = time.perf_counter()
    with ExitStack() as resources:
        pools = [resources.enter_context(ProcessPoolExecutor(max_workers=1, mp_context=mp.get_context('spawn'))) for _ in slots]
        futures = [pools[index].submit(GpuCondenseShard, solver, slot, tuple(str(path) for path in archives[index::len(slots)])) for index, slot in enumerate(slots)]
        completed = [future.result() for future in futures]
    elapsed = time.perf_counter() - beginning
    responses = sorted([item for result, _ in completed for item in result], key=lambda item: item[0])
    return elapsed, responses, [metadata for _, metadata in completed]


def validate_gpu_responses(responses: list[tuple[str, Any]]) -> dict[str, Any]:
    """Check every original source/face row and physical gauge after timed return."""
    import numpy as np

    rows = []
    for name, response in responses:
        columns = np.column_stack((response.source, response.lifts))
        control = original_rows(response.problem, columns)
        retained = response.retained_basis.astype(np.longdouble)
        original_moments = response.problem.constraints.astype(np.longdouble).T @ response.problem.coarse_basis.astype(np.longdouble)
        moment_defect = response.problem.constraints.astype(np.longdouble).T @ (retained-response.problem.coarse_basis.astype(np.longdouble))
        assert np.linalg.norm(moment_defect) <= 1e-10 * np.linalg.norm(original_moments)
        rows.append({'archive': name, 'dofs': response.problem.matrix.shape[0], 'nnz': response.problem.matrix.nnz, 'source_plus_trace_rhs': columns.shape[1], 'represented_kernel_correction': response.problem._correct_kernel, 'executed_rhs_width': columns.shape[1] + (response.problem.kernel.shape[1] if response.problem._correct_kernel else 0), 'max_physical_relative_residual': max(control['physical_relative_residuals']), 'max_augmented_relative_residual': max(control['augmented_relative_residuals']), 'max_gauge_absolute_defect': max(control['gauge_absolute_defects']), 'retained_moment_defect_norm': float(np.linalg.norm(moment_defect)), 'retained_basis_original_action_norm': float(np.linalg.norm(response.problem.matrix @ retained)), 'retained_basis_sha256': hashlib.sha256(np.ascontiguousarray(response.retained_basis).tobytes()).hexdigest()})
    return {'all_original_physical_columns_accepted': True, 'rtol': 1e-10, 'atol': 0.0, 'local_rows': rows}


As with CPU spawn, export only the displayed GPU worker and loader definitions. Its temporary module imports no CUDA package in the coordinator. The independent original-row checks run after the component timer, and include each source/trace column and its physical moment.


In [ ]:
RUN_GPU_COMPONENT = False
GPU_DEVICES = (0, 1)
GPU_REPETITIONS = 3
GPU_WORKER_IMPORTS = """from __future__ import annotations
import os
from pathlib import Path
import time
from typing import Any
import numpy as np
from scipy import sparse
from pymhm.core.contracts import LocalProblem
"""

if RUN_GPU_COMPONENT:
    if last_mhm is None:
        raise ValueError("Execute a CPU formulation first to provide original local operators")
    gpu_source = GPU_WORKER_IMPORTS + "\n" + literal_definitions(("load_problem", "condense_shard")) + "\n"
    gpu_digest = hashlib.sha256(gpu_source.encode()).hexdigest()
    gpu_module_name = "pymhm_darcy3d_gpu_" + gpu_digest[:16]
    gpu_module_path = Path(spawn_directory.name) / (gpu_module_name + ".py")
    gpu_module_path.write_text(gpu_source)
    gpu_specification = importlib.util.spec_from_file_location(gpu_module_name, gpu_module_path)
    gpu_module = importlib.util.module_from_spec(gpu_specification)
    sys.modules[gpu_module_name] = gpu_module
    gpu_specification.loader.exec_module(gpu_module)
    GpuCondenseShard = gpu_module.condense_shard
    local_directory = OUTPUT / "original_local_operators"
    local_directory.mkdir(exist_ok=True)
    for cell in range(len(last_mhm[0].responses)):
        archive_local(last_mhm[0], local_directory / f"cell-{cell:03d}.npz", cell)
    original_archives = tuple(sorted(local_directory.glob("cell-*.npz")))
    gpu_records = []
    for repetition in range(GPU_REPETITIONS):
        seconds, responses, device_metadata = run_gpu_component(original_archives, "amgx", GPU_DEVICES)
        gpu_records.append({"repetition": repetition, "full_component_seconds": seconds,
            "devices": list(GPU_DEVICES), "worker_metadata": device_metadata,
            "scope": "original local condensation component; no FEM or global solve",
            "controls": validate_gpu_responses(responses), "worker_source_sha256": gpu_digest})
    (OUTPUT / "gpu_component.json").write_text(json.dumps(gpu_records, indent=2) + "\n")
    sys.modules.pop(gpu_module_name, None)


## 13. Interpret CPU and GPU comparisons

For a complete speedup against the classical reference, match the physical
problem, quadrature, fine-element budget and stated accuracy, then report the
actual resources for both methods. Equal fine-cell counts alone do not imply
equal approximation error or equal compute resources. Fixed macro/trace
spaces can leave an accuracy plateau when only local fine cells are refined.

The optional named local solver `"amgx"` uses the same Neumann projection and
physical-moment owner as CPU AMG. GPU setup, matrix/RHS upload, all solves,
download, synchronization and cleanup belong to its measured scope. The generic
AmgX adapter serializes native access within one process; selecting a thread
count does not itself distribute work over two GPUs. A multiGPU acquisition
must declare device ownership and transfers explicitly. Resident hierarchy
reuse is a different experiment from the fresh-setup timings above.

For multi-node applications, a process pool on one host is insufficient evidence.
PyMHM's distributed assembly interface provides rank-local ownership and global
PETSc algebra; its scaling requires a separate measured MPI campaign.

The architecture of independent local response construction and a separate
coupled solve follows [Gomes et al. (2017)](https://arxiv.org/abs/1703.10435).
Their published 3D measurements use a different coefficient, tetrahedral P2
spaces, face partitions and cluster resources. This hexahedral analytical
application is not a reproduction of that timing table.
[Penna et al., *Concurrency and Computation*](https://doi.org/10.1002/cpe.5170)
address cost-aware scheduling for heterogeneous MHM workloads. Uniform local
meshes and one-shot Darcy solves do not inherit those measured gains.

See the [3D case description](../../docs/cases/darcy-3d-scalability.md),
[performance guide](../../docs/performance.md) and the
[public numerical record](../../benchmarks/results/execution/introduction-3d-20261004/README.md).


In [ ]:
# Every assemble call has joined its workers before temporary-module cleanup.
sys.path.remove(spawn_directory.name)
sys.modules.pop(module_name, None)
spawn_directory.cleanup()
